In [1]:
# Instalar Optuna en caso de que la sesión de Colab no lo tenga
!pip install -q optuna xgboost lightgbm

import pandas as pd
import numpy as np
import os
import warnings
import joblib

# Optuna para optimización bayesiana
import optuna

# Herramientas de Scikit-Learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Modelos clásicos y avanzados
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING) # Evita que Colab se llene de texto por cada intento
print("✅ Todas las librerías cargadas y listas")

# Montar Drive (si no está montado)
from google.colab import drive
drive.mount('/content/drive')



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 2.7 MB/s eta 0:00:00
✅ Todas las librerías cargadas y listas
Mounted at /content/drive


In [2]:
# ============================================================================
# LOCALIZAR Y CARGAR EL CSV
# ============================================================================

print("="*60)
print("BUSCANDO TU ARCHIVO CSV")
print("="*60)

# Buscar en Drive
csv_files = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.csv') and ('twitter' in file.lower() or 'profile' in file.lower()):
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath)
            csv_files.append((filepath, size))
            print(f"📄 Encontrado: {filepath} ({size:,} bytes)")

if csv_files:
    # Usar el archivo más grande (dataset completo)
    filepath = csv_files[0][0]
    df = pd.read_csv(filepath)
    print(f"\n✅ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    print(f"\n📊 Columnas disponibles:")
    print(df.columns.tolist())
else:
    # Si no encuentra, subir manualmente
    print("❌ No se encontró el archivo. Por favor, súbelo manualmente:")
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)
    print(f"✅ Dataset cargado: {df.shape}")

BUSCANDO TU ARCHIVO CSV
📄 Encontrado: /content/drive/MyDrive/twitter_project/data/processed/twitter_profiles_cleaned.csv (1,486,016 bytes)

✅ Dataset cargado: 7689 filas, 16 columnas

📊 Columnas disponibles:
['name', 'screen_name', 'followers_count', 'friends_count', 'post_count', 'lang', 'location', 'default_profile_image', 'profile_use_background_image', 'verified', 'description', 'created_at', 'label', 'has_location', 'location_clean', 'is_real_location']


In [3]:
# ============================================================================
# PREPARACIÓN DE DATOS (MANTENIENDO TU LÓGICA DE TWITTER)
# ============================================================================

# Seleccionar tus columnas de características
feature_columns = ['followers_count', 'friends_count', 'post_count', 'has_location']
if 'is_real_location' in df.columns:
    feature_columns.append('is_real_location')

# Rellenar nulos de forma segura en las columnas seleccionadas
for col in feature_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(0)

X = df[feature_columns].values
y = df['label'].values

# Dividir en entrenamiento y prueba (80% / 20%) de forma estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Datos listos. Entrenamiento: {X_train.shape[0]} muestras. Prueba: {X_test.shape[0]} muestras.")

✅ Datos listos. Entrenamiento: 6151 muestras. Prueba: 1538 muestras.


In [4]:
# ============================================================================
# DETECTAR LA "TRAMPA" EN LOS DATOS (CORRELACIÓN)
# ============================================================================
print("🕵️‍♂️ Analizando correlación con la variable 'label'...")
correlaciones = df[feature_columns + ['label']].corr()['label'].sort_values(ascending=False)
print(correlaciones)

print("\n💡 Si alguna variable tiene una correlación cercana a 1.0 o -1.0 (ej. 0.98),")
print("esa es la variable 'trampa' que está inflando tus modelos. ¡Debes eliminarla de feature_columns!")

🕵️‍♂️ Analizando correlación con la variable 'label'...
label               1.000000
is_real_location    0.380438
has_location        0.302673
friends_count       0.013174
followers_count    -0.031983
post_count         -0.257472
Name: label, dtype: float64

💡 Si alguna variable tiene una correlación cercana a 1.0 o -1.0 (ej. 0.98),
esa es la variable 'trampa' que está inflando tus modelos. ¡Debes eliminarla de feature_columns!


In [9]:
def objective(trial, X, y):
    steps = []

    # 1. PASO OBLIGATORIO PARA TUS DATOS: Escalado estándar
    steps.append(('scaler', StandardScaler()))

    # 2. PCA CONDICIONAL: Optuna decide si ayuda a reducir ruido en las métricas de Twitter
    use_pca = trial.suggest_categorical("use_pca", [True, False])
    if use_pca:
        pca_variance = trial.suggest_categorical("pca_variance", [0.85, 0.90, 0.95])
        steps.append(('pca', PCA(n_components=pca_variance, random_state=42)))

    # 3. SELECCIÓN DE MODELO FIXED (ALINEADO A INSTRUCCIÓN DOCENTE):
    # Forzamos a que el clasificador sea LightGBM para que Optuna use el 100%
    # de los intentos buscando la combinación perfecta para este algoritmo.
    classifier_name = "LightGBM"

    # Espacio de búsqueda óptimo enfocado únicamente en LightGBM
    lgb_n_estimators = trial.suggest_int("lgb_n_estimators", 50, 150)
    lgb_lr = trial.suggest_float("lgb_learning_rate", 0.01, 0.2, log=True)

    model = LGBMClassifier(
        n_estimators=lgb_n_estimators,
        learning_rate=lgb_lr,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    # Acoplamos el clasificador elegido al pipeline
    steps.append(('classifier', model))
    pipeline = Pipeline(steps)

    # 4. VALIDACIÓN CRUZADA: Evaluamos el pipeline completo de forma segura
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    # Evaluamos con 'f1' para proteger el rendimiento contra desbalances (Bots vs Reales)
    score = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)

    return score.mean()

In [10]:
print("🚀 Iniciando la búsqueda de Hiperparámetros óptimos para LightGBM con Optuna...")

# Creamos el estudio buscando maximizar el F1-Score promedio de la Validación Cruzada
# Añadimos un sampler con semilla fija para blindar el notebook contra cualquier aleatoriedad
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=30)

print("\n" + "="*60)
print(f"👑 PIPELINE GANADOR ENCONTRADO DEFINITIVO:")
print(f"📊 Mejor F1-Score (CV): {study.best_value:.4f}")
print(f"⚙️ Parámetros óptimos: {study.best_params}")
print("="*60 + "\n")

# Guardar los mejores parámetros de forma segura
os.makedirs("outputs/models", exist_ok=True)
joblib.dump(study.best_params, "outputs/models/best_params.pkl")
print("💾 Parámetros guardados con éxito en 'outputs/models/best_params.pkl'")

🚀 Iniciando la búsqueda de Hiperparámetros óptimos para LightGBM con Optuna...

👑 PIPELINE GANADOR ENCONTRADO DEFINITIVO:
📊 Mejor F1-Score (CV): 0.9828
⚙️ Parámetros óptimos: {'use_pca': False, 'lgb_n_estimators': 128, 'lgb_learning_rate': 0.19614548807932547}

💾 Parámetros guardados con éxito en 'outputs/models/best_params.pkl'


In [12]:
# ============================================================================
# EVALUACIÓN FINAL CON EL CONJUNTO DE PRUEBA (DATOS NO VISTOS) - CORREGIDO
# ============================================================================
print("🔄 Reconstruyendo el pipeline óptimo de LightGBM para la evaluación final...")

# Definir la ruta base de Google Drive de manera explícita
BASE_PATH = '/content/drive/MyDrive/twitter_project'
os.makedirs(f"{BASE_PATH}/outputs/models", exist_ok=True)
os.makedirs(f"{BASE_PATH}/data/processed", exist_ok=True)

best_params = study.best_params
final_steps = [('scaler', StandardScaler())]

# 1. Reconstruir el paso de PCA si Optuna determinó que ayudaba
if best_params["use_pca"]:
    final_steps.append(('pca', PCA(n_components=best_params["pca_variance"], random_state=42)))

# 2. Reconstruir directamente el modelo LightGBM Ganador (Simplificado sin IF-ELIF)
# Añadimos manualmente la clave 'classifier' al diccionario para que el Notebook 06
# sepa de manera automatizada qué algoritmo se utilizó.
best_params["classifier"] = "LightGBM"

final_model = LGBMClassifier(
    n_estimators=best_params["lgb_n_estimators"],
    learning_rate=best_params["lgb_learning_rate"],
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

final_steps.append(('classifier', final_model))
best_pipeline = Pipeline(final_steps)

# 3. Entrenar en todo el conjunto de entrenamiento limpio
best_pipeline.fit(X_train, y_train)

# 4. Predecir en el conjunto de prueba real
y_pred_test = best_pipeline.predict(X_test)

# 5. Mostrar reporte de métricas realistas
print("\n📊 REPORTE DE CLASIFICACIÓN FINAL DE PRODUCTION (Datos de Prueba):")
print("-" * 75)
print(classification_report(y_test, y_pred_test, target_names=['Real Account', 'Bot']))

# ============================================================================
# GUARDADO SEGURO DE ARTEFACTOS EN GOOGLE DRIVE
# ============================================================================
print("\n" + "="*60)
print("💾 GUARDANDO DATOS Y MODELOS EN GOOGLE DRIVE...")
print("="*60)

# Guardar los mejores parámetros e hiperparámetros (Incluyendo el nombre del clasificador)
joblib.dump(best_params, f"{BASE_PATH}/outputs/models/best_params.pkl")
print("✅ Hiperparámetros guardados en: outputs/models/best_params.pkl")

# Guardar el pipeline entrenado (el "cerebro" del modelo)
joblib.dump(best_pipeline, f"{BASE_PATH}/outputs/models/best_pipeline_model.pkl")
print("✅ Pipeline final guardado en: outputs/models/best_pipeline_model.pkl")

# Exportar los conjuntos de datos de prueba a formato CSV
pd.DataFrame(X_test).to_csv(f"{BASE_PATH}/data/processed/X_test.csv", index=False)
pd.DataFrame(y_test).to_csv(f"{BASE_PATH}/data/processed/y_test.csv", index=False)

print("✅ X_test.csv guardado con éxito en: data/processed/")
print("✅ y_test.csv guardado con éxito en: data/processed/")
print("\n🎉 ¡Modelos y parámetros sincronizados perfectamente en Drive!")

🔄 Reconstruyendo el pipeline óptimo de LightGBM para la evaluación final...

📊 REPORTE DE CLASIFICACIÓN FINAL DE PRODUCTION (Datos de Prueba):
---------------------------------------------------------------------------
              precision    recall  f1-score   support

Real Account       0.99      0.99      0.99      1008
         Bot       0.98      0.97      0.97       530

    accuracy                           0.98      1538
   macro avg       0.98      0.98      0.98      1538
weighted avg       0.98      0.98      0.98      1538


💾 GUARDANDO DATOS Y MODELOS EN GOOGLE DRIVE...
✅ Hiperparámetros guardados en: outputs/models/best_params.pkl
✅ Pipeline final guardado en: outputs/models/best_pipeline_model.pkl
✅ X_test.csv guardado con éxito en: data/processed/
✅ y_test.csv guardado con éxito en: data/processed/

🎉 ¡Modelos y parámetros sincronizados perfectamente en Drive!
